# Лекция 16: простой Colab по Delta-Neutral RL

Что делаем:
1. Берем свечи Spot/Futures с KuCoin.
2. Считаем basis и z-score.
3. Сравниваем baseline и RL (PPO).
4. Сохраняем JSON для терминала.


## 1) Установка

In [ ]:
!pip -q install numpy pandas matplotlib requests gymnasium stable-baselines3

## 2) Импорт и параметры

In [ ]:
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

SPOT_SYMBOL = "NEAR-USDT"
FUT_SYMBOL = "NEARUSDTM"
TF_MIN = 1
LIMIT = 1200

BASIS_WINDOW = 60
ENTRY_Z = 1.5
EXIT_Z = 0.3

CONTRACT_SIZE = 0.1
MAX_SPOT_NOTIONAL_USDT = 1.0
FEE_BPS = 8.0


## 3) Загрузка OHLCV с KuCoin

In [ ]:
def load_spot(symbol="NEAR-USDT", candle_type="1min"):
    url = "https://api.kucoin.com/api/v1/market/candles"
    rows = requests.get(url, params={"symbol": symbol, "type": candle_type}, timeout=20).json()["data"]
    df = pd.DataFrame(rows, columns=["ts", "open", "close", "high", "low", "volume", "turnover"])
    df["ts"] = df["ts"].astype("int64")
    for c in ["open", "close", "high", "low", "volume", "turnover"]:
        df[c] = df[c].astype("float64")
    return df.sort_values("ts").tail(LIMIT * 2).reset_index(drop=True)


def load_futures(symbol="NEARUSDTM", granularity=1):
    url = "https://api-futures.kucoin.com/api/v1/kline/query"
    rows = requests.get(url, params={"symbol": symbol, "granularity": granularity}, timeout=20).json()["data"]
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "volume", "turnover"])
    df["ts"] = df["ts"].astype("int64")
    df["ts"] = np.where(df["ts"] > 10**12, df["ts"] // 1000, df["ts"])
    for c in ["open", "close", "high", "low", "volume", "turnover"]:
        df[c] = df[c].astype("float64")
    return df.sort_values("ts").tail(LIMIT * 2).reset_index(drop=True)

spot = load_spot(SPOT_SYMBOL, "1min")
fut = load_futures(FUT_SYMBOL, TF_MIN)

df = pd.merge(
    spot[["ts", "close"]].rename(columns={"close": "spot_close"}),
    fut[["ts", "close"]].rename(columns={"close": "fut_close"}),
    on="ts",
    how="inner",
).sort_values("ts").tail(LIMIT).reset_index(drop=True)

print("rows:", len(df))
df.tail(3)


## 4) Basis и z-score

In [ ]:
df["basis"] = (df["fut_close"] - df["spot_close"]) / df["spot_close"]
df["basis_mean"] = df["basis"].rolling(BASIS_WINDOW).mean()
df["basis_std"] = df["basis"].rolling(BASIS_WINDOW).std(ddof=0)
df["basis_z"] = (df["basis"] - df["basis_mean"]) / (df["basis_std"] + 1e-8)

df["spread_bps"] = 10000 * (df["fut_close"] - df["spot_close"]) / df["spot_close"]
df = df.dropna().reset_index(drop=True)

plt.figure(figsize=(12, 4))
plt.plot(df["basis_z"], label="basis_z")
plt.axhline(ENTRY_Z, ls="--", c="green", label="entry_z")
plt.axhline(EXIT_Z, ls=":", c="orange", label="exit_z")
plt.axhline(-EXIT_Z, ls=":", c="orange")
plt.grid(True)
plt.legend()
plt.title("Basis z-score")
plt.show()


## 5) Baseline: простые правила

In [ ]:
def target_pair(spot_price):
    contracts = int(round((MAX_SPOT_NOTIONAL_USDT / spot_price) / CONTRACT_SIZE))
    contracts = max(contracts, 1)
    spot_qty = contracts * CONTRACT_SIZE
    return spot_qty, contracts


def run_baseline(data):
    fee_rate = FEE_BPS / 10000
    position = 0
    spot_qty = 0.0
    fut_contracts = 0
    equity = 0.0
    rows = []

    for i in range(len(data) - 1):
        row = data.iloc[i]
        nxt = data.iloc[i + 1]
        z = float(row["basis_z"])
        prev_position = position
        fee = 0.0

        if position == 0 and z > ENTRY_Z:
            spot_qty, fut_contracts = target_pair(float(row["spot_close"]))
            position = 1
            fee += fee_rate * (spot_qty * row["spot_close"] + fut_contracts * CONTRACT_SIZE * row["fut_close"])

        elif position == 1 and abs(z) < EXIT_Z:
            fee += fee_rate * (spot_qty * row["spot_close"] + fut_contracts * CONTRACT_SIZE * row["fut_close"])
            position = 0
            spot_qty = 0.0
            fut_contracts = 0

        pnl = 0.0
        if position == 1:
            fut_base = -fut_contracts * CONTRACT_SIZE
            pnl += spot_qty * (nxt["spot_close"] - row["spot_close"])
            pnl += fut_base * (nxt["fut_close"] - row["fut_close"])

        reward = pnl - fee
        equity += reward

        rows.append({
            "ts": int(row["ts"]),
            "position": int(position),
            "position_change": int(position != prev_position),
            "step_reward": float(reward),
            "equity": float(equity),
        })

    return pd.DataFrame(rows)

baseline = run_baseline(df)
baseline.tail(3)


## 6) RL-среда и PPO

In [ ]:
class DeltaNeutralEnv(gym.Env):
    def __init__(self, data):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.action_space = spaces.Discrete(3)  # 0 hold, 1 open/keep, 2 close
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32)
        self.reset()

    def _obs(self):
        row = self.data.iloc[self.i]
        return np.array([
            float(row["basis_z"]),
            float(row["basis"]),
            float(row["spread_bps"]),
            float(self.position),
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.i = 0
        self.position = 0
        self.spot_qty = 0.0
        self.fut_contracts = 0
        self.equity = 0.0
        return self._obs(), {}

    def step(self, action):
        row = self.data.iloc[self.i]
        nxt = self.data.iloc[self.i + 1]
        fee_rate = FEE_BPS / 10000

        prev_position = self.position
        fee = 0.0

        if action == 1 and self.position == 0 and float(row["basis_z"]) > 0:
            self.spot_qty, self.fut_contracts = target_pair(float(row["spot_close"]))
            self.position = 1
            fee += fee_rate * (self.spot_qty * row["spot_close"] + self.fut_contracts * CONTRACT_SIZE * row["fut_close"])

        if action == 2 and self.position == 1:
            fee += fee_rate * (self.spot_qty * row["spot_close"] + self.fut_contracts * CONTRACT_SIZE * row["fut_close"])
            self.position = 0
            self.spot_qty = 0.0
            self.fut_contracts = 0

        pnl = 0.0
        if self.position == 1:
            fut_base = -self.fut_contracts * CONTRACT_SIZE
            pnl += self.spot_qty * (nxt["spot_close"] - row["spot_close"])
            pnl += fut_base * (nxt["fut_close"] - row["fut_close"])

        reward = pnl - fee
        self.equity += reward

        self.i += 1
        done = self.i >= len(self.data) - 1

        info = {
            "ts": int(row["ts"]),
            "position": int(self.position),
            "position_change": int(self.position != prev_position),
            "step_reward": float(reward),
            "equity": float(self.equity),
        }

        obs = self._obs() if not done else np.zeros(4, dtype=np.float32)
        return obs, float(reward), done, False, info

split = int(len(df) * 0.7)
train_df = df.iloc[:split].reset_index(drop=True)
test_df = df.iloc[split:].reset_index(drop=True)

vec_env = DummyVecEnv([lambda: DeltaNeutralEnv(train_df)])
model = PPO("MlpPolicy", vec_env, verbose=0, seed=SEED, n_steps=256, batch_size=256)
model.learn(total_timesteps=25000)
print("RL ready")


## 7) Сравнение baseline vs RL

In [ ]:
def run_rl(model, data):
    env = DeltaNeutralEnv(data)
    obs, _ = env.reset()
    rows = []
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        done = terminated or truncated
        rows.append(info)
    return pd.DataFrame(rows)


def metrics(bt):
    eq = bt["equity"].values
    r = bt["step_reward"].values
    dd = eq - np.maximum.accumulate(eq)
    return {
        "total_pnl": float(eq[-1]),
        "max_drawdown": float(dd.min()),
        "sharpe_step": float(np.mean(r) / (np.std(r) + 1e-9)),
        "trades": int(bt["position_change"].sum()),
    }

baseline_test = run_baseline(test_df)
rl_test = run_rl(model, test_df)

print("Baseline:", metrics(baseline_test))
print("RL PPO  :", metrics(rl_test))

plt.figure(figsize=(12, 5))
plt.plot(baseline_test["equity"].values, label="Baseline")
plt.plot(rl_test["equity"].values, label="RL PPO")
plt.grid(True)
plt.legend()
plt.title("Equity Curve")
plt.show()


## 8) Экспорт JSON для терминала

In [ ]:
latest = df.iloc[-1]

state_json = {
    "signal_model": "basis_zscore",
    "timestamp": int(latest["ts"]),
    "spot_price": float(latest["spot_close"]),
    "futures_price": float(latest["fut_close"]),
    "basis": float(latest["basis"]),
    "basis_mean": float(latest["basis_mean"]),
    "basis_std": float(latest["basis_std"]),
    "basis_z": float(latest["basis_z"]),
    "entry_z": float(ENTRY_Z),
    "exit_z": float(EXIT_Z),
    "mode": "long_spot_short_futures_only",
}

out_path = "/content/latest_forecast_signal_kucoin_rl.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(state_json, f, ensure_ascii=False, indent=2)

print(json.dumps(state_json, ensure_ascii=False, indent=2))


In [ ]:
from google.colab import files
files.download("/content/latest_forecast_signal_kucoin_rl.json")


## 9) Команда для терминала

```powershell
python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json "$HOME\Downloads\latest_forecast_signal_kucoin_rl.json"
```
